# Notebook 05 — Mean Reversion & Pairs Trading

**Phase 2 · Strategy Modules (2 / 4)**

---

## 🎯 Learning Objectives

| # | Objective |
|---|----------|
| 1 | Understand the mean-reversion hypothesis: *what goes down must come back up* |
| 2 | Build a Bollinger Band + RSI signal strength indicator from scratch |
| 3 | Understand cointegration and the Augmented Dickey-Fuller (ADF) test |
| 4 | Compute OLS hedge ratios and mean-reversion half-life |
| 5 | Construct z-score-based entry / exit rules for pairs trading |
| 6 | Compare scratch code to `evaluate_mean_reversion_signal()` and `find_cointegrated_pairs()` |

### Prerequisites
- NB02 (RSI, Bollinger Bands)
- NB04 (momentum — the *opposite* strategy)

In [ ]:
# ── Boilerplate ────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
print("✅ imports ready  |  project root:", ROOT)

---
## 1 · The Mean-Reversion Hypothesis

While **momentum** bets that winners keep winning, **mean reversion** bets on the opposite: assets that have fallen "too far" bounce back.

Two complementary entry signals:

| Signal | Trigger | Intuition |
|--------|---------|----------|
| **RSI oversold** | RSI ≤ 30 | Selling pressure exhausted |
| **Bollinger Band breach** | Price ≤ lower band | Price is > 2σ below its mean |

Our production code combines both into a single **signal strength** ∈ [0, 1]:

$$
\text{RSI\_signal} = \text{clip}\!\left(\frac{\text{RSI}_{\text{oversold}} - \text{RSI}}{\text{RSI}_{\text{oversold}}},\; 0,\; 1\right)
$$
$$
\text{BB\_signal} = \text{clip}\!\left(\frac{\text{LowerBand} - P}{P} \times 20,\; 0,\; 1\right)
$$
$$
\text{strength} = \max(\text{RSI\_signal},\; \text{BB\_signal})
$$

A strength of 0 means "no signal"; 1 means a very strong oversold reading.

---
## 2 · Synthetic Data

In [ ]:
np.random.seed(42)
DAYS = 120
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=DAYS, freq="D")

# Asset with a crash-and-recovery (ideal for mean reversion)
crash_start, crash_end = 60, 80
base_drift = np.random.normal(0.001, 0.02, DAYS)
base_drift[crash_start:crash_end] = np.random.normal(-0.03, 0.02, crash_end - crash_start)
base_drift[crash_end:crash_end+10] = np.random.normal(0.02, 0.015, 10)

eth_prices = 3500 * np.exp(np.cumsum(base_drift))
eth_volumes = np.random.uniform(10e6, 60e6, DAYS)

prices = pd.Series(eth_prices, index=dates, name="ETHUSDT")
quote_volumes = pd.Series(eth_volumes, index=dates, name="ETHUSDT")

print(f"Price range: {prices.min():.0f} – {prices.max():.0f}")
prices.plot(title="ETHUSDT – Synthetic Crash & Recovery", ylabel="Price (USD)")
plt.show()

---
## 3 · Building the Mean-Reversion Indicator Frame

### 3.1  Bollinger Bands + RSI

In [ ]:
# ── RSI (from NB02) ───────────────────────────────────────
def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    delta = prices.astype(float).diff()
    gains = delta.clip(lower=0.0)
    losses = (-delta).clip(lower=0.0)
    avg_gain = gains.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = losses.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss_safe = avg_loss.mask(avg_loss == 0.0)
    rs = avg_gain / avg_loss_safe
    rsi = 100.0 - (100.0 / (1.0 + rs))
    rsi = rsi.mask((avg_loss == 0.0) & (avg_gain > 0.0), 100.0)
    rsi = rsi.mask((avg_gain == 0.0) & (avg_loss > 0.0), 0.0)
    rsi = rsi.mask((avg_gain == 0.0) & (avg_loss == 0.0), 50.0)
    return rsi.astype(float)

# ── Bollinger Bands ───────────────────────────────────────
BB_PERIOD = 20
BB_STD    = 2.0
RSI_OVERSOLD = 30.0

ma = prices.rolling(BB_PERIOD).mean()
std = prices.rolling(BB_PERIOD).std(ddof=0)
upper_band = ma + BB_STD * std
lower_band = ma - BB_STD * std
rsi = calculate_rsi(prices)

# Signal strengths
rsi_signal = ((RSI_OVERSOLD - rsi) / RSI_OVERSOLD).clip(lower=0.0, upper=1.0)
bb_signal  = (((lower_band - prices) / prices) * 20.0).clip(lower=0.0, upper=1.0)
strength   = pd.concat([rsi_signal, bb_signal], axis=1).max(axis=1)

print(f"Max signal strength: {strength.max():.3f}")
print(f"Days with strength > 0: {(strength > 0).sum()}")

### 3.2  Visualising the Signal

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# ── Panel 1: Price + Bollinger Bands ──
ax = axes[0]
ax.plot(prices.index, prices, label="Close", lw=1.5)
ax.plot(ma.index, ma, label="MA(20)", ls="--", lw=1)
ax.fill_between(prices.index, lower_band, upper_band, alpha=0.15, color="blue", label="Bollinger Band")
# Highlight oversold entries
entry_mask = strength > 0
ax.scatter(prices.index[entry_mask], prices[entry_mask],
           color="red", s=20, zorder=5, label="Signal ON")
ax.set_ylabel("Price")
ax.legend(loc="upper left", fontsize=8)
ax.set_title("Price with Bollinger Bands & Mean-Reversion Signals")

# ── Panel 2: RSI ──
ax2 = axes[1]
ax2.plot(rsi.index, rsi, color="purple", lw=1.2)
ax2.axhline(RSI_OVERSOLD, color="green", ls="--", lw=1, label="Oversold = 30")
ax2.axhline(70, color="red", ls="--", lw=1, label="Overbought = 70")
ax2.fill_between(rsi.index, 0, RSI_OVERSOLD, alpha=0.1, color="green")
ax2.set_ylabel("RSI")
ax2.set_ylim(0, 100)
ax2.legend(loc="upper left", fontsize=8)

# ── Panel 3: Signal Strength ──
ax3 = axes[2]
ax3.fill_between(strength.index, 0, strength, color="orange", alpha=0.6)
ax3.set_ylabel("Signal Strength")
ax3.set_ylim(0, 1.1)
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

plt.tight_layout()
plt.show()

---
## 4 · Production Code: `evaluate_mean_reversion_signal()`

Compare our scratch indicator with the production function.

In [ ]:
from bot.signals.mean_reversion import (
    build_mean_reversion_frame,
    evaluate_mean_reversion_signal,
    MeanReversionSignal,
)

# Build the full frame
prod_frame = build_mean_reversion_frame(
    prices, quote_volumes,
    rsi_period=14,
    rsi_oversold=30.0,
    bollinger_period=20,
    bollinger_std=2.0,
    volume_window=24,
)

print("Production indicator frame columns:", prod_frame.columns.tolist())
prod_frame.tail(5)

In [ ]:
# Get the latest signal
signal = evaluate_mean_reversion_signal(
    prices, quote_volumes,
    rsi_period=14,
    rsi_oversold=30.0,
    bollinger_period=20,
    bollinger_std=2.0,
    min_volume_usd=10_000_000.0,
)

if signal is not None:
    print(f"Signal detected!")
    print(f"  strength    = {signal.strength:.3f}")
    print(f"  price       = {signal.price:.2f}")
    print(f"  MA          = {signal.moving_average:.2f}")
    print(f"  lower_band  = {signal.lower_band:.2f}")
    print(f"  RSI         = {signal.rsi:.1f}")
    print(f"  volume_24h  = {signal.volume_24h:,.0f}")
else:
    print("No mean-reversion signal at the latest bar (price may not be oversold).")
    print(f"  Latest RSI: {rsi.iloc[-1]:.1f}  (threshold: {RSI_OVERSOLD})")
    print(f"  Latest price: {prices.iloc[-1]:.2f}  Lower band: {lower_band.iloc[-1]:.2f}")

---
## 5 · Pairs Trading: The Theory

### 5.1  What Is Cointegration?

Two price series $P_A$ and $P_B$ are **cointegrated** if there exists a constant $\beta$ (hedge ratio) such that the spread:

$$
S_t = P_{A,t} - \beta \cdot P_{B,t}
$$

is **stationary** — it fluctuates around a constant mean and reverts to it.  This is *stronger* than correlation: correlated assets move together, but cointegrated assets are *bound together*.

### 5.2  The Trading Logic

Standardise the spread into a **z-score**:

$$
z_t = \frac{S_t - \bar{S}}{\sigma_S}
$$

| Condition | Action |
|-----------|--------|
| $z < -z_{\text{entry}}$ | Spread too low → **long A, short B** |
| $z > +z_{\text{entry}}$ | Spread too high → **short A, long B** |
| $|z| < z_{\text{exit}}$ | Spread reverted → **close position** |

Our default: $z_{\text{entry}} = 2.0$.

---
## 6 · Step-by-Step: Pairs Trading from Scratch

### 6.1  Create Cointegrated Synthetic Pairs

In [ ]:
np.random.seed(123)
n = 200
dates_pairs = pd.date_range(end=pd.Timestamp.now().normalize(), periods=n, freq="D")

# Create a pair with a known cointegrating relationship
# B is the driver, A = beta*B + mean-reverting noise
BETA_TRUE = 1.5
SPREAD_MEAN = 50.0
SPREAD_STD  = 10.0

b_returns = np.random.normal(0.002, 0.03, n)
price_b = 100 * np.exp(np.cumsum(b_returns))

# Mean-reverting spread (Ornstein-Uhlenbeck process)
theta = 0.1   # speed of reversion
noise_std = 3.0
spread = np.zeros(n)
spread[0] = SPREAD_MEAN
for t in range(1, n):
    spread[t] = spread[t-1] + theta * (SPREAD_MEAN - spread[t-1]) + np.random.normal(0, noise_std)

price_a = BETA_TRUE * price_b + spread

pair_closes = pd.DataFrame({
    "ASSET_A": price_a,
    "ASSET_B": price_b,
}, index=dates_pairs)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pair_closes.plot(ax=axes[0], title="Cointegrated Pair – Price Series")
axes[1].plot(spread, color="orange", lw=1.2)
axes[1].axhline(SPREAD_MEAN, color="gray", ls="--")
axes[1].set_title(f"True Spread (mean={SPREAD_MEAN})")
axes[1].set_ylabel("Spread")
plt.tight_layout()
plt.show()

### 6.2  OLS Hedge Ratio

In [ ]:
def compute_hedge_ratio(series_a: pd.Series, series_b: pd.Series) -> float:
    """OLS regression: A = beta * B + intercept."""
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        series_b.values.astype(float),
        series_a.values.astype(float),
    )
    return float(slope)

estimated_beta = compute_hedge_ratio(pair_closes["ASSET_A"], pair_closes["ASSET_B"])
print(f"True β  = {BETA_TRUE:.3f}")
print(f"Est. β  = {estimated_beta:.3f}")
print(f"Error   = {abs(estimated_beta - BETA_TRUE):.4f}")

### 6.3  Compute Spread & ADF Test

In [ ]:
# Estimated spread
est_spread = pair_closes["ASSET_A"] - estimated_beta * pair_closes["ASSET_B"]

# Simple ADF: regress ΔS on S_{t-1}
def simple_adf_pvalue(spread: pd.Series) -> float:
    """Run a simple ADF test via OLS regression."""
    lagged = spread.shift(1).iloc[1:]
    delta  = spread.diff().iloc[1:]
    if lagged.std() == 0:
        return 1.0
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        lagged.values.astype(float),
        delta.values.astype(float),
    )
    return float(p_value)

adf_p = simple_adf_pvalue(est_spread)
print(f"ADF p-value: {adf_p:.4f}")
print(f"Cointegrated at 5% level: {'YES ✅' if adf_p < 0.05 else 'NO ❌'}")

### 6.4  Half-Life of Mean Reversion

From an AR(1) model $\Delta S_t = \phi \cdot S_{t-1} + \varepsilon$, the half-life is:

$$
h = -\frac{\ln 2}{\phi}
$$

A half-life of 5 days means the spread typically reverts halfway to its mean in 5 days.

In [ ]:
def estimate_half_life(spread: pd.Series) -> float:
    """Estimate mean-reversion half-life using AR(1)."""
    lagged = spread.shift(1).dropna()
    delta  = spread.diff().dropna()
    common = lagged.index.intersection(delta.index)
    if len(common) < 3:
        return float("inf")
    slope, _, _, _, _ = stats.linregress(
        lagged.loc[common].values.astype(float),
        delta.loc[common].values.astype(float),
    )
    if slope >= 0:
        return float("inf")  # not mean-reverting
    return float(-np.log(2) / slope)

hl = estimate_half_life(est_spread)
print(f"Estimated half-life: {hl:.1f} days")
print(f"(True θ = {theta}, implied theoretical half-life ≈ {-np.log(2)/np.log(1-theta):.1f} days)")

### 6.5  Z-Score & Trading Signals

In [ ]:
# Z-score
spread_mean = est_spread.mean()
spread_std  = est_spread.std()
z_score = (est_spread - spread_mean) / spread_std

Z_ENTRY = 2.0
Z_EXIT  = 0.5

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# ── Spread ──
ax = axes[0]
ax.plot(est_spread.index, est_spread, lw=1.2, label="Estimated Spread")
ax.axhline(spread_mean, color="gray", ls="--", label=f"Mean = {spread_mean:.1f}")
ax.axhline(spread_mean + Z_ENTRY * spread_std, color="red", ls=":", label=f"+{Z_ENTRY}σ")
ax.axhline(spread_mean - Z_ENTRY * spread_std, color="green", ls=":", label=f"-{Z_ENTRY}σ")
ax.set_ylabel("Spread")
ax.legend(fontsize=8)
ax.set_title("Spread & Entry Thresholds")

# ── Z-Score ──
ax2 = axes[1]
ax2.plot(z_score.index, z_score, lw=1.2, color="purple")
ax2.axhline(Z_ENTRY, color="red", ls=":", label=f"Entry = ±{Z_ENTRY}")
ax2.axhline(-Z_ENTRY, color="green", ls=":")
ax2.axhline(Z_EXIT, color="orange", ls="--", alpha=0.5, label=f"Exit = ±{Z_EXIT}")
ax2.axhline(-Z_EXIT, color="orange", ls="--", alpha=0.5)
ax2.axhline(0, color="gray", lw=0.5)
ax2.fill_between(z_score.index, -Z_ENTRY, Z_ENTRY, alpha=0.05, color="gray")
ax2.set_ylabel("Z-Score")
ax2.legend(fontsize=8)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

plt.tight_layout()
plt.show()

### 6.6  Simulated Pairs Trade PnL

In [ ]:
# ── Simple pairs trading backtest ─────────────────────────
position = 0  # +1 = long spread, -1 = short spread, 0 = flat
pnl_list = []
trades = []

for i in range(1, len(z_score)):
    z = z_score.iloc[i]
    spread_return = est_spread.iloc[i] - est_spread.iloc[i-1]
    
    # PnL from existing position
    pnl = position * spread_return
    pnl_list.append({"date": z_score.index[i], "pnl": pnl, "position": position, "z": z})
    
    # Entry / Exit logic
    if position == 0:
        if z < -Z_ENTRY:
            position = 1   # long spread (buy A, sell B)
            trades.append((z_score.index[i], "LONG SPREAD", z))
        elif z > Z_ENTRY:
            position = -1  # short spread (sell A, buy B)
            trades.append((z_score.index[i], "SHORT SPREAD", z))
    elif position == 1 and z > -Z_EXIT:
        position = 0
        trades.append((z_score.index[i], "EXIT LONG", z))
    elif position == -1 and z < Z_EXIT:
        position = 0
        trades.append((z_score.index[i], "EXIT SHORT", z))

pnl_df = pd.DataFrame(pnl_list).set_index("date")
pnl_df["cumulative_pnl"] = pnl_df["pnl"].cumsum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pnl_df.index, pnl_df["cumulative_pnl"], lw=1.5, color="#2ecc71")
ax.set_ylabel("Cumulative PnL (spread units)")
ax.set_title("Pairs Trading Backtest – Cumulative PnL")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout()
plt.show()

print(f"Total trades: {len(trades)}")
print(f"Final PnL: {pnl_df['cumulative_pnl'].iloc[-1]:.2f}")
for dt, action, z in trades[:10]:
    print(f"  {dt.strftime('%Y-%m-%d')}  {action:15s}  z={z:+.2f}")

---
## 7 · Production Code: `find_cointegrated_pairs()`

Our production code automates the entire pipeline: screen all $(N \choose 2)$ pairs, test cointegration, estimate half-life, and return scored `PairSignal` objects.

In [ ]:
from bot.signals.pairs_rotation import (
    find_cointegrated_pairs,
    pairs_rotation_weights,
    PairSignal,
)

# Create a multi-asset panel for pair screening
np.random.seed(77)
SYMBOLS_PAIR = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "ADAUSDT", "XRPUSDT", "DOTUSDT"]
n_pair = 120
dates_p = pd.date_range(end=pd.Timestamp.now().normalize(), periods=n_pair, freq="D")

# Make some pairs cointegrated by construction
driver = np.cumsum(np.random.normal(0.001, 0.02, n_pair))
multi_closes = pd.DataFrame(index=dates_p)
multi_closes["BTCUSDT"] = 60000 * np.exp(driver)
multi_closes["ETHUSDT"] = 3500 * np.exp(driver * 1.3 + np.cumsum(np.random.normal(0, 0.005, n_pair)))
multi_closes["SOLUSDT"] = 140 * np.exp(np.cumsum(np.random.normal(0.003, 0.04, n_pair)))
multi_closes["ADAUSDT"] = 0.45 * np.exp(np.cumsum(np.random.normal(-0.001, 0.03, n_pair)))
multi_closes["XRPUSDT"] = 0.55 * np.exp(driver * 0.8 + np.cumsum(np.random.normal(0, 0.008, n_pair)))
multi_closes["DOTUSDT"] = 7 * np.exp(np.cumsum(np.random.normal(0.002, 0.035, n_pair)))

# Find cointegrated pairs
pair_signals = find_cointegrated_pairs(
    multi_closes,
    lookback=60,
    adf_threshold=0.05,
    min_half_life=1.0,
    max_half_life=30.0,
)

print(f"Found {len(pair_signals)} cointegrated pairs:\n")
for ps in pair_signals:
    print(f"  {ps.asset_a} / {ps.asset_b}")
    print(f"    z-score={ps.z_score:+.2f}  hedge_ratio={ps.hedge_ratio:.4f}")
    print(f"    half_life={ps.half_life:.1f}d  ADF p={ps.adf_pvalue:.4f}")
    print()

In [ ]:
# ── Convert pair signals to portfolio weights ─────────────
weights = pairs_rotation_weights(pair_signals, z_entry=2.0, max_pairs=3)

print("Pairs rotation weight adjustments:")
for sym, w in sorted(weights.items(), key=lambda x: x[1], reverse=True):
    print(f"  {sym:10s}  {w:+.3f}")

if not weights:
    print("  (no actionable pairs — all z-scores within ±2.0)")

---
## 8 · `PairSignal` Anatomy

```python
@dataclass(frozen=True, slots=True)
class PairSignal:
    asset_a: str         # first asset in the pair
    asset_b: str         # second asset
    z_score: float       # current standardised spread
    spread: float        # raw spread value
    half_life: float     # estimated days to revert 50%
    hedge_ratio: float   # OLS β for the spread
    adf_pvalue: float    # ADF test p-value (< 0.05 = cointegrated)
```

**Key design choices:**
- Sorted by `|z_score|` descending — most extreme pairs first.
- `pairs_rotation_weights()` converts signals into portfolio weight *adjustments* (can be negative for short legs).
- The `max_pairs=3` cap prevents excessive concentration.

---
## 9 · Mean Reversion vs Momentum: When to Use Each?

| Factor | Momentum | Mean Reversion |
|--------|----------|---------------|
| **Market regime** | Strong trend (bull/bear) | Ranging / choppy |
| **Time horizon** | Medium (days-weeks) | Short (hours-days) |
| **Risk profile** | Momentum crashes | Catching falling knives |
| **Best pairs with** | Trend confirmation (EMA) | Cointegration test |

This is exactly why our bot uses a **regime-adaptive ensemble** — the regime detector (NB06) decides which strategy gets more weight.

In a **bull** regime: momentum weight ↑, mean reversion weight ↓  
In a **ranging** regime: momentum weight ↓, mean reversion weight ↑

---
## 10 · Key Takeaways

| Concept | Detail |
|---------|--------|
| **Signal strength** | max(RSI_signal, BB_signal) ∈ [0, 1] |
| **Cointegration** | Two series bound by a stationary spread (ADF p < 0.05) |
| **Hedge ratio** | OLS slope β so that A - βB is stationary |
| **Half-life** | AR(1)-based estimate of reversion speed |
| **Z-score entry** | |z| > 2.0 → trade; |z| < 0.5 → exit |
| **Config** | `strategy_params.yaml` → `mean_reversion:` section |

---
## 🔬 Exercises

1. **Strength decomposition:** Modify the signal to use a *weighted average* of RSI and BB signals instead of `max()`. Does `0.6 * rsi_signal + 0.4 * bb_signal` produce more consistent entries?

2. **Half-life filter:** In the pairs backtest, skip any pair with a half-life > 10 days. Does this improve PnL?

3. **Rolling cointegration:** Cointegration can break down over time. Implement a rolling 60-day window ADF test and plot the p-value over time for the BTCUSDT / ETHUSDT pair.

4. **Stop-loss for pairs:** Add a stop-loss to the pairs backtest: if the z-score moves 1.5x further against you after entry (e.g., entered at z = -2, exit if z reaches -5), close the position. Does this reduce drawdowns?

---
## ✅ Knowledge Check

1. What is the difference between *correlation* and *cointegration*?
2. What does a half-life of 7 days mean intuitively?
3. Why does the production code use `max(RSI_signal, BB_signal)` instead of summing them?
4. What happens if the ADF p-value is 0.50?  Should you trade that pair?
5. How does `pairs_rotation_weights()` handle a z-score of -3.0 (i.e., z < -z_entry)?

---
## 🔗 Next

**[NB06 — Regime Detection →](06_Regime_Detection.ipynb)**

We'll learn how the bot classifies the market into **bull / ranging / bear** regimes — the key that determines *which* strategy gets the most weight.